# SeedVR2（现代 Web 视频与图片 4K 超分工作台 + 全参数 API）

- **支持显卡**：40GB 及以上显存显卡（`A100 40GB / 80GB`、`H100 80GB`、`RTX 6000 96GB`）
- **单页双模态超分**：在同一工作台切换视频超分（`7B Sharp` / `3B Fast`）与图片超分；可指定 4K 目标尺寸，实际尺寸取决于原图和放大策略
- **后处理与显存控制**：内置 **CIELAB / Wavelet / AdaIN** 色彩对齐、**Split Latent** 时间分块（支持单帧 / `4n+1` 分块与 Hann 窗融合）、**Tiled VAE** 编解码与 **同屏 A/B 卷帘对比**；保留原声内容，但音轨随成片重新编码，不保证位流无损
- **访问方式**：默认使用 Cloudflare `Quick Tunnel (ephemeral)` 临时公网域名（用完即关）；亦可在第 4 步切换 `Custom Tunnel (CF_TUNNEL_TOKEN)` 从 Colab Secrets 读取 `CF_TUNNEL_TOKEN` 挂载固定域名
- **执行顺序**：`1. 环境检测` → `2. 安装内部推理引擎与应用服务包` → `3. 下载并校验 SeedVR2 双规格权重与 VAE` → `4. 启动现代 Web 工作台与 API 服务`

| 入口路径 | 说明 |
| --- | --- |
| `GET /` | 现代亮色视频/图片超分工作台（含按倍数/短边档位/长边/精确宽高换算、视频与图片 A/B 卷帘对比及会话历史） |
| `POST /api/v1/generate` | 异步提交视频超分或图片 4K 超分修复任务（返回 `task_id`） |
| `POST /api/v1/files` | 上传待超分的源视频（MP4/WebM/MOV/MKV）或源图片（PNG/JPG/WebP） |
| `GET /api/v1/tasks/{task_id}` | 查询实时推理进度与超分产物地址（亦支持 `WS /api/v1/tasks/{task_id}/ws`） |
| `GET /docs` | Swagger / OpenAPI 在线交互接口文档 |

### 1. 环境检测
- **功能**：检测 GPU 型号、显存容量、BF16 支持、系统内存、磁盘空间及 PyTorch / CUDA 版本。
- **要求**：支持 `A100`、`H100`、`RTX 6000`，显存 40GB 及以上。
- **使用**：直接运行，输出 `[OK]` 后继续。

In [ ]:
# @title 1. Environment Check {display-mode: "form"}
import os, shutil, subprocess, torch

MIN_VRAM_GIB = 35
REQUIREMENT = "40GB+ VRAM (A100 40GB/80GB, H100 80GB, RTX 6000 96GB)"

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Select Runtime -> Change runtime type -> A100 / H100 / RTX 6000 GPU.")

print(subprocess.run(["nvidia-smi"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True).stdout)

gpu = torch.cuda.get_device_name(0)
vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
ram_gib = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
disk_gib = shutil.disk_usage("/content").free / 1024**3
bf16 = torch.cuda.is_bf16_supported()
print(f"GPU: {gpu} | VRAM: {vram_gib:.1f} GiB | RAM: {ram_gib:.1f} GiB | Disk Free: {disk_gib:.1f} GiB | BF16: {bf16}")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")

if not any(k in gpu.upper() for k in ("A100", "H100", "6000")) or vram_gib < MIN_VRAM_GIB:
    raise RuntimeError(f"Detected {gpu} ({vram_gib:.1f} GiB), but this notebook requires {REQUIREMENT}. Please switch runtime.")
if not bf16:
    raise RuntimeError("Current GPU does not support BF16; please switch runtime.")
print(f"[OK] GPU environment ready: {gpu} ({vram_gib:.1f} GiB)")

### 2. 安装内部推理引擎与应用服务包
- **功能**：安装内部无头 GPU 推理算子引擎（ComfyUI `v0.43.2`）、FastAPI 服务依赖、`cloudflared` `2026.10.6`，并拉取校验 GitHub Release 发布的 `app-seedvr2.tar.gz`（`v0.2.3`，内含编译好的现代前端 `dist` 与全参数 API 服务）。
- **使用**：直接运行，输出 `[OK]` 后继续。

In [ ]:
# @title 2. Install Inference Engine & App Bundle {display-mode: "form"}
import hashlib, re, shutil, subprocess, sys, tarfile, tempfile, urllib.request
from pathlib import Path

COMFY_DIR = Path("/content/ComfyUI")
COMFY_REPO = "https://github.com/Comfy-Org/ComfyUI"
COMFY_VERSION = "v0.43.2"
COMFY_REVISION = "423cbd20511becb1f16122ab518e92e58bfa03d9"

APP_DIR = Path("/content/app")
RELEASE_REPO = "boltguo/colab"
RELEASE_TAG = "v0.2.3"
APP_ARCHIVE = "app-seedvr2.tar.gz"
APP_SHA256 = "fe21526efb5d0ef3ab298a3324e95a4949d60e589f1dce1f8a744769d2a7153f"
BUNDLED_NODES_REL = ""
CLOUDFLARED_VERSION = "2026.10.6"

EXTRA_PACKAGES = ["av>=19.0.1"]
APP_SERVER_PACKAGES = [
    "fastapi>=0.115.0,<1.0.0",
    "uvicorn[standard]>=0.30.0,<1.0.0",
    "python-multipart>=0.0.9,<1.0.0",
    "pydantic>=2.7.0,<3.0.0",
    "huggingface_hub>=0.23.0",
    "hf_xet",
]

def run(args, cwd=None):
    subprocess.run(args, cwd=cwd, check=True)

def git(path, *args):
    return subprocess.check_output(["git", *args], cwd=path, text=True).strip()

def ensure_not_symlink(path, label):
    for p in [path, *path.parents]:
        if p == Path("/content"):
            break
        if p.is_symlink():
            raise RuntimeError(f"{label} path contains a symlink; start a fresh runtime: {p}")

def checkout_repo(path, repo_url, revision, label, allow_local_changes=False):
    ensure_not_symlink(path, label)
    fresh = not path.exists()
    if fresh:
        path.parent.mkdir(parents=True, exist_ok=True)
        run(["git", "clone", "--depth", "1", "--no-checkout", repo_url + ".git", str(path)])
    if not (path / ".git").is_dir():
        raise RuntimeError(f"{label} is not a git repository; please use a clean runtime: {path}")
    if git(path, "remote", "get-url", "origin").rstrip("/").removesuffix(".git") != repo_url:
        raise RuntimeError(f"Unexpected remote origin in {label}; please use a clean runtime.")
    if not fresh and not allow_local_changes and git(path, "status", "--porcelain"):
        raise RuntimeError(f"{label} has uncommitted local changes; please use a clean runtime.")
    if fresh or git(path, "rev-parse", "HEAD") != revision:
        run(["git", "fetch", "--depth", "1", "origin", revision], cwd=path)
        run(["git", "checkout", "--force", "--detach", revision], cwd=path)

def sync_bundled_custom_nodes():
    if not BUNDLED_NODES_REL:
        return
    src_root = APP_DIR / BUNDLED_NODES_REL
    if not src_root.is_dir():
        return
    custom_nodes_dir = COMFY_DIR / "custom_nodes"
    ensure_not_symlink(custom_nodes_dir, "ComfyUI custom_nodes")
    custom_nodes_dir.mkdir(parents=True, exist_ok=True)
    for node_dir in sorted(p for p in src_root.iterdir() if p.is_dir()):
        if node_dir.is_symlink() or any(child.is_symlink() for child in node_dir.rglob("*")):
            raise RuntimeError(f"Refusing symlink inside bundled custom nodes: {node_dir}")
        dest = custom_nodes_dir / node_dir.name
        if dest.is_symlink():
            raise RuntimeError(f"Destination custom node path is a symlink: {dest}")
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(node_dir, dest, symlinks=False)
        print(f"Synced bundled ComfyUI node pack: {node_dir.name}")

def install_release_app():
    ensure_not_symlink(APP_DIR, "App directory")
    stamp_file = APP_DIR / ".release_sha256"
    if (
        stamp_file.is_file()
        and not stamp_file.is_symlink()
        and stamp_file.read_text(encoding="utf-8").strip() == f"{RELEASE_TAG}:{APP_SHA256}"
        and (APP_DIR / "backend/run_server.py").is_file()
    ):
        print(f"App bundle already up to date: {APP_ARCHIVE} ({RELEASE_TAG})")
        return

    local_archive = Path(f"/content/{APP_ARCHIVE}")
    with tempfile.NamedTemporaryFile(suffix=".tar.gz", delete=False) as tmp:
        tmp_path = Path(tmp.name)
    try:
        if local_archive.is_file() and not local_archive.is_symlink():
            print(f"Using pre-staged app archive: {local_archive}")
            payload = local_archive.read_bytes()
        else:
            url = f"https://github.com/{RELEASE_REPO}/releases/download/{RELEASE_TAG}/{APP_ARCHIVE}"
            print(f"Downloading app bundle: {url}")
            req = urllib.request.Request(url, headers={"User-Agent": "colab-release-fetcher"})
            with urllib.request.urlopen(req, timeout=60) as resp:
                payload = resp.read()
        actual_sha = hashlib.sha256(payload).hexdigest()
        if actual_sha != APP_SHA256:
            raise RuntimeError(
                f"App bundle SHA256 mismatch! Expected {APP_SHA256}, got {actual_sha}"
            )
        tmp_path.write_bytes(payload)

        staging = APP_DIR.with_name("app.staging")
        if staging.is_symlink():
            raise RuntimeError("Staging directory is a symlink; aborting.")
        if staging.exists():
            shutil.rmtree(staging)
        staging.mkdir(parents=True, exist_ok=True)
        staging_root = staging.resolve()

        with tarfile.open(tmp_path, "r:gz") as tf:
            for member in tf.getmembers():
                if member.issym() or member.islnk():
                    raise RuntimeError(f"Refusing symlink/hardlink archive entry: {member.name}")
                target = (staging / member.name).resolve()
                if target != staging_root and not str(target).startswith(str(staging_root) + "/"):
                    raise RuntimeError(f"Archive path escapes staging root: {member.name}")
            tf.extractall(staging)

        (staging / ".release_sha256").write_text(f"{RELEASE_TAG}:{APP_SHA256}\n", encoding="utf-8")
        if APP_DIR.exists():
            shutil.rmtree(APP_DIR)
        staging.replace(APP_DIR)
    finally:
        if tmp_path.exists():
            tmp_path.unlink()

checkout_repo(COMFY_DIR, COMFY_REPO, COMFY_REVISION, "ComfyUI")
REQUIREMENT_FILES = [COMFY_DIR / "requirements.txt"]

skip = {"torch", "torchvision", "torchaudio", "xformers"}
lines = []
for req_file in REQUIREMENT_FILES:
    for line in req_file.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        name = re.split(r"[<>=!~\[;\s]", s, maxsplit=1)[0].strip().lower()
        if name not in skip:
            lines.append(s)
requirements = Path("/content/comfy_requirements.txt")
requirements.write_text("\n".join(lines) + "\n", encoding="utf-8")
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements), *EXTRA_PACKAGES, *APP_SERVER_PACKAGES])

install_release_app()
sync_bundled_custom_nodes()

cloudflared_bin = Path("/usr/local/bin/cloudflared")
try:
    installed = subprocess.run([str(cloudflared_bin), "--version"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True).stdout
except OSError:
    installed = ""
if CLOUDFLARED_VERSION not in installed.split():
    download = cloudflared_bin.with_name("cloudflared.download")
    urllib.request.urlretrieve(
        f"https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-amd64",
        str(download),
    )
    download.chmod(0o755)
    download.replace(cloudflared_bin)

print(f"Engine: ComfyUI {COMFY_VERSION} ({COMFY_REVISION[:12]}) | app: {RELEASE_TAG} ({APP_SHA256[:12]}) | cloudflared: {CLOUDFLARED_VERSION}")
print("[OK] Internal inference engine and Web/API bundle installed.")

### 3. 下载并校验 SeedVR2 双规格权重与 VAE
- **功能**：高速下载并精确校验 SeedVR2 FP16 `7B Sharp`、`3B Fast` 与 `EMA VAE`（约 22.1 GB）至 `/content/ComfyUI/models`（启动后可在 Web 工作台或 API 中自由切换 `7B Sharp` 旗舰锐化版与 `3B Fast` 长视频极速版）。
- **使用**：直接运行，输出 `[OK]` 后继续。

In [ ]:
# @title 3. Download & Verify Base Model Weights {display-mode: "form"}
import os, shutil, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")
os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")
os.environ.setdefault("HF_XET_CHUNK_CACHE_SIZE_BYTES", "0")
from huggingface_hub import hf_hub_download

COMFY_DIR = Path("/content/ComfyUI")
MODEL_DIR = COMFY_DIR / "models"
SEEDVR2 = ("Comfy-Org/SeedVR2", "df48879708206a403d2a61acd55578c2e80fd233", "")

MODEL_FILES = [
    (*SEEDVR2, "diffusion_models/seedvr2_7b_sharp_fp16.safetensors", 16480583960),
    (*SEEDVR2, "diffusion_models/seedvr2_3b_fp16.safetensors", 6784268336),
    (*SEEDVR2, "vae/seedvr2_ema_vae_fp16.safetensors", 501324814),
]

def local_directory(path):
    path = Path(path)
    for parent in [path, *path.parents]:
        if parent == COMFY_DIR.parent:
            break
        if parent.is_symlink():
            raise RuntimeError(f"Directory is a symlink; please start a fresh runtime: {parent}")
    path.mkdir(parents=True, exist_ok=True)

def model_ready(target, size):
    return target.is_file() and not target.is_symlink() and target.stat().st_size == size

def download_model(repo, revision, folder, file, size):
    base = MODEL_DIR / folder if folder else MODEL_DIR
    target = base / file
    local_directory(target.parent)
    started = time.time()
    result = Path(
        hf_hub_download(
            repo_id=repo,
            filename=file,
            revision=revision,
            local_dir=str(base),
            force_download=target.exists(),
            token=False,
        )
    )
    if result.resolve() != target.resolve() or not model_ready(target, size):
        raise RuntimeError(f"Incomplete or mismatched model download: {target}")
    print(f"Downloaded: {file} ({size / 1024**3:.2f} GiB in {time.time() - started:.0f}s)")

def download_models():
    pending = []
    for item in MODEL_FILES:
        base = MODEL_DIR / item[2] if item[2] else MODEL_DIR
        target = base / item[3]
        if target.is_symlink():
            raise RuntimeError(f"Refusing symlink model file: {target}")
        if model_ready(target, item[4]):
            print("Verified:", item[3])
        else:
            pending.append(item)
    if not pending:
        return
    missing = sum(item[4] for item in pending)
    free = shutil.disk_usage(MODEL_DIR).free
    if free < missing + 2 * 1024**3:
        raise RuntimeError(f"Insufficient disk space: need {missing / 1024**3:.1f} GiB more, free {free / 1024**3:.1f} GiB.")
    started = time.time()
    pool = ThreadPoolExecutor(max_workers=3)
    try:
        futures = [pool.submit(download_model, *item) for item in sorted(pending, key=lambda item: -item[4])]
        for future in as_completed(futures):
            future.result()
    finally:
        pool.shutdown(wait=False, cancel_futures=True)
    elapsed = time.time() - started
    print(f"Base weights ready: {missing / 1024**3:.1f} GiB in {elapsed:.0f}s ({missing / 1024**2 / max(elapsed, 1):.0f} MB/s)")

for path in [MODEL_DIR, MODEL_DIR / "loras", COMFY_DIR / "input", COMFY_DIR / "output"]:
    local_directory(path)
download_models()
print("[OK] Base model weights verified.")

### 4. 启动现代 Web 工作台与 API 服务
- **功能**：启动 FastAPI 服务（`127.0.0.1:8188`，同时承载 `/` 现代前端工作台、`/api/v1/*` 业务接口与 `/docs` OpenAPI 文档），并挂载 Cloudflare Tunnel。
- **隧道选择**：`TUNNEL_MODE` 默认选择 `Quick Tunnel (ephemeral)`（免费公网临时域名，实例关闭即失效）；也可切换为 `Custom Tunnel (CF_TUNNEL_TOKEN)`，从 Colab 左侧 **🔑 Secrets** 读取 `CF_TUNNEL_TOKEN` 挂载固定域名。
- **访问密码**：在下方 `ACCESS_PASSWORD` 填写应用访问密码，或在 Secrets 配置同名项。API 默认免鉴权；需要鉴权时，登录网页并在右上角「API Key」中单独设置密钥。共享笔记本时建议使用 Secrets。
- **使用**：直接运行，点击输出的公网链接即可在网页操作或直接复制对应链接调用 API。

In [ ]:
# @title 4. Launch Web Studio & API Service {display-mode: "form"}
TUNNEL_MODE = "Quick Tunnel (ephemeral)"  # @param ["Quick Tunnel (ephemeral)", "Custom Tunnel (CF_TUNNEL_TOKEN)"]
ACCESS_PASSWORD = ""  # @param {type:"string"}
USE_CF_TUNNEL_TOKEN = TUNNEL_MODE.startswith("Custom")
import hashlib, json, os, re, shutil, subprocess, sys, time, urllib.request
from pathlib import Path

FEATURE_NAME = "seedvr2"
BUNDLED_NODES_REL = ""
COMFY_DIR = Path("/content/ComfyUI")
COMFY_PORT = 8189
COMFY_BASE = f"http://127.0.0.1:{COMFY_PORT}"
COMFY_LOG = Path("/content/comfyui-worker.log")
COMFY_LAUNCHER = Path("/content/comfy_userdata_compat.py")
COMFY_LAUNCH_CONFIG = Path("/content/comfyui-worker.launch.json")
COMFY_PROCESS_PATTERN = r"comfy_userdata_compat\.py|ComfyUI/main\.py"
COMFY_LAUNCH_ARGS = [
    "--output-directory", str(COMFY_DIR / "output"),
    "--disable-all-custom-nodes",
    "--disable-partner-nodes",
]
REQUIRED_NODES = {"SeedVR2Preprocess", "SeedVR2PostProcessing", "SeedVR2Conditioning", "SeedVR2TemporalChunk", "SeedVR2TemporalMerge", "ResizeImageMaskNode", "SaveVideo"}

APP_DIR = Path("/content/app")
DATA_DIR = Path(f"/content/app_data/{FEATURE_NAME}")
PORT = 8188
BASE = f"http://127.0.0.1:{PORT}"
APP_LOG = Path(f"/content/{FEATURE_NAME}-web.log")
APP_LAUNCH_CONFIG = Path(f"/content/{FEATURE_NAME}.launch.json")
APP_PROCESS_PATTERN = r"backend/run_server\.py|backend\.run_server"

CF_BIN = Path("/usr/local/bin/cloudflared")
CF_LOG = Path("/content/cloudflared.log")

def read_text(path):
    return path.read_text(errors="replace") if path.exists() else ""

def tail(path, size=8000):
    return read_text(path)[-size:]

def rotate_log(path):
    if path.is_file() and path.stat().st_size:
        path.replace(path.with_name(path.name + ".1"))

def process_running(pattern):
    return subprocess.run(["pgrep", "-f", pattern], stdout=subprocess.DEVNULL).returncode == 0

def stop_processes(pattern, timeout):
    subprocess.run(["pkill", "-f", pattern], check=False)
    deadline = time.time() + timeout
    while process_running(pattern) and time.time() < deadline:
        time.sleep(2)
    if process_running(pattern):
        subprocess.run(["pkill", "-9", "-f", pattern], check=False)
        time.sleep(2)

def resolve_cf_token():
    t = ""
    try:
        from google.colab import userdata
        t = (userdata.get("CF_TUNNEL_TOKEN") or "").strip()
    except Exception:
        t = ""
    if not t:
        t = os.environ.get("CF_TUNNEL_TOKEN", "").strip()
    return t.split()[-1] if t else ""

def start_cf_tunnel(token=""):
    if not CF_BIN.exists():
        raise RuntimeError("cloudflared not found; please run Step 2 first.")
    stop_processes("cloudflared tunnel", 10)
    rotate_log(CF_LOG)
    cmd = [
        str(CF_BIN),
        "tunnel",
        "--no-autoupdate",
        "--protocol", "http2",
        "--edge-ip-version", "4",
    ]
    if token:
        cmd.extend(["run", "--url", BASE, "--token", token])
    else:
        cmd.extend(["--url", BASE])
    with CF_LOG.open("w") as cf_log:
        cf_proc = subprocess.Popen(
            cmd,
            stdin=subprocess.DEVNULL,
            stdout=cf_log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    deadline = time.time() + (30 if token else 60)
    while time.time() < deadline:
        if cf_proc.poll() is not None:
            print(tail(CF_LOG, 4000))
            if token:
                raise RuntimeError("Cloudflare Tunnel failed to start; check CF_TUNNEL_TOKEN.")
            raise RuntimeError("Cloudflare Quick Tunnel failed to start; check logs above.")
        log_text = read_text(CF_LOG)
        if not token:
            match = re.search(r"https://(?!api\.)[a-zA-Z0-9-]+\.trycloudflare\.com", log_text)
            if match and "Registered tunnel connection" in log_text:
                url = match.group(0)
                print(f"[OK] Cloudflare Quick Tunnel connected (HTTP/2): {url} -> {BASE}")
                return url
        elif "Registered tunnel connection" in log_text:
            if "Updated to new configuration" not in log_text:
                time.sleep(1)
                log_text = read_text(CF_LOG)
            configs = [line for line in log_text.splitlines() if "Updated to new configuration" in line]
            config = configs[-1].replace('\\"', '"') if configs else ""
            routes = re.findall(r'"hostname"\s*:\s*"([^"]+)"(?:(?!"hostname").)*?"service"\s*:\s*"([^"]+)"', config)
            for host, service in routes:
                print(f"[OK] Cloudflare Tunnel connected (HTTP/2): https://{host} -> {service}")
            hosts = [host for host, service in routes if f":{PORT}" in service]
            if not routes:
                print("[OK] Cloudflare Tunnel connected (HTTP/2); open via your configured Cloudflare hostname.")
            elif not hosts:
                print(f"[WARN] No public hostname forwards to port {PORT}!")
                print(f"       Set Service URL to {BASE} under Cloudflare Zero Trust -> Networks -> Tunnels -> Public Hostname")
            return f"https://{hosts[0]}" if hosts else "https://<your-cf-domain>"
        time.sleep(1)
    if not token:
        match = re.search(r"https://(?!api\.)[a-zA-Z0-9-]+\.trycloudflare\.com", read_text(CF_LOG))
        if match:
            url = match.group(0)
            print(f"[OK] Cloudflare Quick Tunnel ready: {url} -> {BASE}")
            return url
        print(tail(CF_LOG, 4000))
        raise RuntimeError(f"Cloudflare Quick Tunnel returned no URL within 60s; see {CF_LOG}.")
    print(f"[WARN] Cloudflare Tunnel connection not confirmed within 30s (still running); see {CF_LOG}.")
    return "https://<your-cf-domain>"

def verify_web_password(base_url, process_pattern):
    from urllib.error import HTTPError

    class NoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, req, fp, code, msg, headers, newurl):
            return None

    opener = urllib.request.build_opener(urllib.request.ProxyHandler({}), NoRedirect())
    try:
        with opener.open(base_url + "/", timeout=10):
            pass
    except HTTPError as error:
        if error.code == 303 and error.headers.get("Location") == "/auth/login":
            print("[OK] Web access password protection verified.")
            return
    except Exception:
        pass
    stop_processes(process_pattern, 30)
    raise RuntimeError("Web access password check failed; stopped web server. Update the notebook and rerun Steps 2 and 4.")

def check_public_url(url, route):
    if "<" in url:
        return
    error = None
    deadline = time.time() + 60
    while time.time() < deadline:
        try:
            request = urllib.request.Request(url + route, headers={"User-Agent": "colab-tunnel-check"})
            with urllib.request.urlopen(request, timeout=10):
                print(f"[OK] Public URL reachable: {url}")
                return
        except Exception as e:
            error = e
        time.sleep(3)
    print(f"[WARN] Public URL unreachable within 60s: {url}{route} ({error})")

cf_token = resolve_cf_token() if USE_CF_TUNNEL_TOKEN else ""
if USE_CF_TUNNEL_TOKEN and not cf_token:
    raise RuntimeError("Custom Tunnel is selected, but CF_TUNNEL_TOKEN was not found. Add CF_TUNNEL_TOKEN in Colab Secrets (or switch to Quick Tunnel).")

if not (COMFY_DIR / "main.py").is_file():
    raise RuntimeError("Internal engine /content/ComfyUI not found; please run Step 2 first.")
if APP_DIR.is_symlink() or not (APP_DIR / "backend/run_server.py").is_file():
    raise RuntimeError("App directory /content/app not found; please run Step 2 first.")

if FEATURE_NAME in ("minimax-h3", "seedvr2", "h3-character-swap"):
    missing_media_tools = [name for name in ("ffmpeg", "ffprobe") if not shutil.which(name)]
    if missing_media_tools:
        raise RuntimeError(f"Missing required media tools: {', '.join(missing_media_tools)}. Run !apt-get -qq update && apt-get -qq install -y ffmpeg first.")

if BUNDLED_NODES_REL:
    bundled_nodes_src = APP_DIR / BUNDLED_NODES_REL
    if bundled_nodes_src.is_dir():
        custom_nodes_dir = COMFY_DIR / "custom_nodes"
        custom_nodes_dir.mkdir(parents=True, exist_ok=True)
        for node_dir in sorted(p for p in bundled_nodes_src.iterdir() if p.is_dir()):
            if node_dir.is_symlink() or any(c.is_symlink() for c in node_dir.rglob("*")):
                raise RuntimeError(f"Refusing symlink inside bundled custom nodes: {node_dir}")
            dest = custom_nodes_dir / node_dir.name
            if dest.is_symlink():
                raise RuntimeError(f"Destination custom node path is a symlink: {dest}")
            if dest.exists():
                shutil.rmtree(dest)
            shutil.copytree(node_dir, dest, symlinks=False)

def resolve_access_password():
    if ACCESS_PASSWORD.strip():
        return ACCESS_PASSWORD.strip()
    key = ""
    try:
        from google.colab import userdata
        key = (userdata.get("ACCESS_PASSWORD") or "").strip()
    except Exception:
        key = ""
    if not key:
        key = os.environ.get("ACCESS_PASSWORD", "").strip()
    return key

access_password = resolve_access_password()
if not access_password:
    raise RuntimeError(
        "Set ACCESS_PASSWORD above or configure Colab Secret `ACCESS_PASSWORD`."
    )

LAUNCHER_SOURCE = r'''
"""Start pinned ComfyUI with narrow workflow aliases and Cloudflare compatibility."""
import os
import runpy
import sys
from pathlib import Path
from aiohttp import web


def add_workflow_aliases(manager, routes):
    originals = [route for route in routes
                 if getattr(route, "path", None) == "/userdata/{file}"
                 and getattr(route, "method", None) in {"GET", "POST", "DELETE"}]
    if {route.method for route in originals} != {"GET", "POST", "DELETE"}:
        raise RuntimeError("ComfyUI userdata routes changed; workflow compatibility aliases not installed.")
    for route in originals:
        async def workflow_alias(request, handler=route.handler):
            file = request.match_info.get("file", "")
            user_root = manager.get_request_user_filepath(request, None, create_dir=False)
            candidate = manager.get_request_user_filepath(request, file, create_dir=False)
            if not user_root or not candidate:
                return web.Response(status=403)
            workflow_directory = os.path.join(user_root, "workflows")
            if os.path.islink(workflow_directory):
                return web.Response(status=403)
            workflow_root = os.path.realpath(workflow_directory)
            try:
                within = os.path.commonpath((workflow_root, os.path.realpath(candidate))) == workflow_root
            except ValueError:
                within = False
            if not within or os.path.realpath(candidate) == workflow_root:
                return web.Response(status=403)
            return await handler(request)
        routes.route(route.method, "/userdata/{file:workflows/[^/]+}")(workflow_alias)


def install_cf_middleware():
    import middleware.cache_middleware as cm
    orig_cache_control = cm.cache_control

    @web.middleware
    async def cf_safe_middleware(request, handler):
        response = await orig_cache_control(request, handler)
        if request.method == "GET" and request.path == "/" and getattr(response, "status", None) == 200:
            import server
            inst = getattr(server.PromptServer, "instance", None)
            web_root = getattr(inst, "web_root", None) if inst else None
            index_file = Path(web_root) / "index.html" if web_root else None
            if index_file and index_file.is_file():
                html = index_file.read_text(encoding="utf-8", errors="replace")
                html = html.replace("<script ", '<script data-cfasync="false" ')
                new_resp = web.Response(text=html, content_type="text/html")
                for k, v in response.headers.items():
                    if k.lower() not in {"content-length", "content-type", "etag", "last-modified"}:
                        new_resp.headers[k] = v
                new_resp.headers["Cache-Control"] = "no-store, no-transform, must-revalidate"
                response = new_resp
        if isinstance(response, web.StreamResponse) and not isinstance(response, web.WebSocketResponse) and not response.prepared:
            if getattr(response, "status", None) == 200 and (
                request.path.startswith("/assets/")
                or request.path.endswith((".woff2", ".woff", ".ttf"))
            ):
                response.headers["Cache-Control"] = "public, max-age=604800, immutable, no-transform"
            else:
                cc = response.headers.get("Cache-Control", "")
                if "no-transform" not in cc:
                    response.headers["Cache-Control"] = f"{cc}, no-transform" if cc else "no-transform"
            origin = request.headers.get("Origin")
            if origin and response.headers.get("Access-Control-Allow-Origin") == "*":
                response.headers["Access-Control-Allow-Origin"] = origin
                response.headers["Vary"] = "Origin"
        return response

    cm.cache_control = cf_safe_middleware


def main():
    comfy_root = Path("/content/ComfyUI")
    if comfy_root.is_symlink() or not (comfy_root / "main.py").is_file() or not (comfy_root / "comfy/options.py").is_file():
        raise RuntimeError("Missing or invalid ComfyUI directory.")
    os.chdir(comfy_root)
    sys.path.insert(0, str(comfy_root))
    sys.argv[0] = str(comfy_root / "main.py")
    import comfy.options
    comfy.options.enable_args_parsing()
    install_cf_middleware()
    from app.user_manager import UserManager
    original = UserManager.add_routes

    def compatible_routes(self, routes):
        original(self, routes)
        add_workflow_aliases(self, routes)

    UserManager.add_routes = compatible_routes
    runpy.run_path("main.py", run_name="__main__")


if __name__ == "__main__":
    main()
'''

def comfy_get_json(route, timeout=5):
    with urllib.request.urlopen(COMFY_BASE + route, timeout=timeout) as response:
        return json.load(response)

def comfy_ready(timeout=5):
    try:
        comfy_get_json("/system_stats", timeout)
        return True
    except Exception:
        return False

# 1. Start internal headless ComfyUI worker (127.0.0.1:8189, loopback only)
comfy_cfg_str = json.dumps({"launcher": LAUNCHER_SOURCE, "args": COMFY_LAUNCH_ARGS, "port": COMFY_PORT})
comfy_running = comfy_ready() or (process_running(COMFY_PROCESS_PATTERN) and comfy_ready(timeout=30))
if comfy_running and read_text(COMFY_LAUNCH_CONFIG) != comfy_cfg_str:
    print("Restarting internal ComfyUI worker with updated launch configuration...")
    stop_processes(COMFY_PROCESS_PATTERN, 60)
    comfy_running = False

if comfy_running:
    print("[OK] Internal ComfyUI worker already running (127.0.0.1:8189).")
else:
    stop_processes(COMFY_PROCESS_PATTERN, 60)
    COMFY_LAUNCHER.write_text(LAUNCHER_SOURCE, encoding="utf-8")
    rotate_log(COMFY_LOG)
    with COMFY_LOG.open("w") as log:
        comfy_proc = subprocess.Popen(
            [sys.executable, str(COMFY_LAUNCHER), "--listen", "127.0.0.1", "--port", str(COMFY_PORT), *COMFY_LAUNCH_ARGS],
            cwd=COMFY_DIR,
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    COMFY_LAUNCH_CONFIG.write_text(comfy_cfg_str, encoding="utf-8")
    print("Starting internal GPU inference engine...")
    deadline = time.time() + 600
    while time.time() < deadline:
        if comfy_proc.poll() is not None:
            print(tail(COMFY_LOG))
            raise RuntimeError(f"Internal engine failed to start (exit code {comfy_proc.returncode}); see logs above.")
        if comfy_ready():
            print("[OK] Internal GPU inference engine ready (127.0.0.1:8189).")
            break
        time.sleep(2)
    else:
        print(tail(COMFY_LOG))
        raise TimeoutError("Internal engine did not become ready within 10 minutes; check logs.")

missing = sorted(REQUIRED_NODES - set(comfy_get_json("/object_info", timeout=60)))
if missing:
    print(tail(COMFY_LOG))
    raise RuntimeError(f"Missing required ComfyUI nodes: {', '.join(missing)}. Re-run Step 2.")

# 2. Start public FastAPI gateway & static Web Studio (127.0.0.1:8188)
def app_ready(timeout=5):
    try:
        with urllib.request.urlopen(BASE + "/api/v1/health", timeout=timeout) as r:
            return r.status == 200
    except Exception:
        return False

env_overrides = {
    "HF_HUB_DISABLE_IMPLICIT_TOKEN": "1",
    "COLAB_FEATURE": FEATURE_NAME,
    "COLAB_HOST": "127.0.0.1",
    "COLAB_PORT": str(PORT),
    "COLAB_DATA_DIR": str(DATA_DIR),
    "COLAB_DIST_DIR": str(APP_DIR / "dist" / FEATURE_NAME),
    "COLAB_ACCESS_PASSWORD": access_password,
    "COLAB_API_KEY": "",
    "COMFY_BASE_URL": COMFY_BASE,
    "COMFY_DIR": str(COMFY_DIR),
}

app_cmd = [
    sys.executable,
    str(APP_DIR / "backend/run_server.py"),
    "--feature",
    FEATURE_NAME,
    "--host",
    "127.0.0.1",
    "--port",
    str(PORT),
    "--dist-dir",
    str(APP_DIR / "dist" / FEATURE_NAME),
]

app_fingerprint = json.dumps({
    "cmd": app_cmd,
    "release": read_text(APP_DIR / ".release_sha256").strip(),
    "web_auth_sha256": hashlib.sha256(access_password.encode("utf-8")).hexdigest(),
    "env": {key: value for key, value in env_overrides.items() if key not in {"COLAB_ACCESS_PASSWORD", "COLAB_API_KEY"}},
}, sort_keys=True)
app_is_up = app_ready() or (process_running(APP_PROCESS_PATTERN) and app_ready(timeout=15))
if app_is_up and read_text(APP_LAUNCH_CONFIG) != app_fingerprint:
    print("Web/API configuration changed; restarting...")
    stop_processes(APP_PROCESS_PATTERN, 30)
    app_is_up = False

if app_is_up:
    print("[OK] Web Studio and API service already running (127.0.0.1:8188).")
else:
    stop_processes(APP_PROCESS_PATTERN, 30)
    rotate_log(APP_LOG)
    child_env = {**os.environ, **env_overrides}
    with APP_LOG.open("w") as log:
        app_proc = subprocess.Popen(
            app_cmd,
            cwd=APP_DIR,
            env=child_env,
            stdin=subprocess.DEVNULL,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    APP_LAUNCH_CONFIG.write_text(app_fingerprint, encoding="utf-8")
    print("Starting Web Studio and API service...")
    deadline = time.time() + 120
    while time.time() < deadline:
        if app_proc.poll() is not None:
            print(tail(APP_LOG))
            raise RuntimeError(f"Web/API service failed to start (exit code {app_proc.returncode}); see logs above.")
        if app_ready():
            print("[OK] Web Studio and API service started.")
            break
        time.sleep(1.5)
    else:
        print(tail(APP_LOG))
        raise TimeoutError("Web/API service did not become ready within 2 minutes; check logs.")

# 3. Expose via Cloudflare Tunnel & persist live URL metadata
verify_web_password(BASE, APP_PROCESS_PATTERN)
host_url = start_cf_tunnel(cf_token)
tunnel_mode_key = "custom" if cf_token else "quick"
if not DATA_DIR.is_symlink():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    tunnel_info_path = DATA_DIR / "tunnel_info.json"
    if not tunnel_info_path.is_symlink():
        tunnel_info_path.write_text(
            json.dumps({"public_url": host_url, "tunnel_mode": tunnel_mode_key}, ensure_ascii=False),
            encoding="utf-8",
        )
check_public_url(host_url, "/api/v1/health")

sample_body = json.dumps({'mode': 'video', 'input_file': '<UPLOADED_FILE_ID>', 'model_variant': '7b_sharp', 'sizing_mode': 'multiplier', 'scale_factor': 2.0, 'color_correction': 'lab', 'keep_audio': True, 'steps': 1}, ensure_ascii=False)
auth_note = ' -H "Authorization: Bearer YOUR_API_KEY"'
tunnel_tip = (
    "WARNING: Quick Tunnel provides a temporary URL that expires when the runtime stops. Select Custom Tunnel above for a permanent domain."
    if tunnel_mode_key == "quick"
    else "INFO: Using your custom fixed Cloudflare Tunnel domain."
)
print(f"""
=== Endpoints ===
Web Studio   : {host_url}
API Docs     : {host_url}/docs
System Stats : {host_url}/api/v1/system
{tunnel_tip}
API Key      : Open by default; configure an optional key in the Web Studio header.

=== Quick API Example ===
curl -X POST "{host_url}/api/v1/generate" \\
  -H "Content-Type: application/json"{auth_note} \\
  -d '{sample_body}'
""")